# SmileAlign v2, lanjutkan latihan di Kaggle

Notebook ini **melanjutkan** latihan yang terputus, bukan memulai dari nol.

## Sebelum mulai

1. Kaggle: **Settings > Accelerator: GPU P100** (atau T4), **Internet: On**.
2. Di Drive, dua file ini harus diset **Anyone with the link**:
   - `smilealign/runs-smilealign-seg-v2/<run>/weights/last.pt`
   - `smilealign/intraoral_yolov8.zip`
3. Ambil ID dari link, bagian antara `/d/` dan `/view`.

## Kenapa bisa lanjut tanpa mengulang

`last.pt` menyimpan nomor epoch, state optimizer, dan semua argumen latihan.
Sel 9 memanggil `train(resume=True)` tanpa argumen lain.

## Tiga hal yang harus persis sama

- **Path dataset.** Checkpoint menyimpan `/content/datasets/mix_v2/data.yaml`.
  Sel 8 membangun dataset di path itu, bukan di `/kaggle/working`.
- **Path folder hasil.** Checkpoint menyimpan
  `/content/drive/MyDrive/smilealign/runs-smilealign-seg-v2`.
  Sel 2 membuat symlink ke `/kaggle/working/smilealign` supaya path itu hidup.
- **Versi Ultralytics.** Checkpoint dibuat dengan versi tertentu. Sel 5 memasang
  versi yang sama persis, dibaca langsung dari isi checkpoint. Ini yang
  sebelumnya gagal karena versi dipin ke 8.3.

## Pakai `last.pt`, bukan `best.pt`

`best.pt` tidak menyimpan state optimizer dan nomor epoch, resume dari situ
mengulang dari epoch 1.

## Sel 0, konfigurasi

In [ ]:
# Tempel URL atau ID Drive, dua-duanya diterima.
# URL  : https://drive.google.com/file/d/1AbCdEfGhIjKlMnOpQrStUvWxYz/view?usp=sharing
# ID   : 1AbCdEfGhIjKlMnOpQrStUvWxYz

FILE_LAST_PT = ""
FILE_INTRAORAL_ZIP = ""

# Kosongkan. Nama folder run dibaca dari dalam checkpoint di Sel 4.
RUN_NAME = ""

import re
from pathlib import Path


def drive_id(value: str) -> str:
    # Terima URL Drive atau ID polos, kembalikan ID polos.
    value = (value or "").strip()
    m = re.search(r"/d/([A-Za-z0-9_-]{10,})", value)
    if m:
        return m.group(1)
    m = re.search(r"[?&]id=([A-Za-z0-9_-]{10,})", value)
    if m:
        return m.group(1)
    return value


LAST_PT_ID = drive_id(FILE_LAST_PT)
ZIP_ID = drive_id(FILE_INTRAORAL_ZIP)

ROOT = Path("/content/drive/MyDrive/smilealign")
RUNS = ROOT / "runs-smilealign-seg-v2"
STAGING = Path("/kaggle/working/_incoming")
MIX_DIR = Path("/content/datasets/mix_v2")

print("last.pt id  :", LAST_PT_ID or "(kosong)")
print("zip id      :", ZIP_ID or "(kosong)")
print("ROOT        :", ROOT)
print("RUNS        :", RUNS)
print("siap        :", bool(LAST_PT_ID and ZIP_ID))

## Sel 1, cek GPU dan pasang paket dasar

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import subprocess
import sys
import torch


def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("GAGAL:", " ".join(specs))
        print(r.stdout[-1200:])
        print(r.stderr[-1200:])
    return r.returncode == 0


# ultralytics sengaja TIDAK dipasang di sini. Versinya harus mengikuti isi
# checkpoint, dan itu baru diketahui setelah Sel 4.
assert pip_install("gdown", "huggingface_hub"), "paket dasar gagal dipasang"

print("torch :", torch.__version__)
print("cuda  :", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU belum aktif, ubah Settings > Accelerator"

## Sel 2, hidupkan path Drive

Checkpoint menyimpan `project: /content/drive/MyDrive/smilealign/runs-smilealign-seg-v2`.
Path itu tidak ada di Kaggle, jadi dibuat symlink ke folder kerja. Hasilnya
Ultralytics menulis ke folder yang sama persis seperti di Colab.

In [ ]:
import os
from pathlib import Path

work = Path("/kaggle/working/smilealign")
work.mkdir(parents=True, exist_ok=True)

Path("/content/drive/MyDrive").mkdir(parents=True, exist_ok=True)
link = Path("/content/drive/MyDrive/smilealign")

if link.is_symlink():
    print("symlink sudah ada ->", os.path.realpath(link))
elif link.exists():
    print("PERINGATAN: ada folder biasa, bukan symlink:", link)
else:
    os.symlink(work, link)
    print("symlink dibuat ->", os.path.realpath(link))

RUNS.mkdir(parents=True, exist_ok=True)
print("isi RUNS:", [p.name for p in RUNS.iterdir()] or "(kosong)")

## Sel 3, unduh last.pt dan zip intraoral

In [ ]:
import gdown
from pathlib import Path

assert LAST_PT_ID and ZIP_ID, "ID belum diisi di Sel 0"

STAGING.mkdir(parents=True, exist_ok=True)

# Cek /kaggle/input dulu, kalau file sudah diunggah sebagai Kaggle Dataset
# maka tidak perlu internet sama sekali.
pre_uploaded = list(Path("/kaggle/input").rglob("intraoral_yolov8.zip"))
zip_path = ROOT / "intraoral_yolov8.zip"

if pre_uploaded and not zip_path.exists():
    import shutil

    shutil.copy2(pre_uploaded[0], zip_path)
    print("zip dari /kaggle/input:", pre_uploaded[0])
elif not zip_path.exists():
    gdown.download(id=ZIP_ID, output=str(zip_path), quiet=False)

print("zip      :", zip_path, round(zip_path.stat().st_size / 1e6, 1), "MB")
assert zip_path.stat().st_size > 1e6, "zip terlalu kecil, kemungkinan gagal unduh"

staged_last = STAGING / "last.pt"
if not staged_last.exists():
    gdown.download(id=LAST_PT_ID, output=str(staged_last), quiet=False)

print("last.pt  :", staged_last, round(staged_last.stat().st_size / 1e6, 1), "MB")
assert staged_last.stat().st_size > 1e6, "last.pt terlalu kecil, kemungkinan gagal unduh"

## Sel 4, periksa checkpoint

Gerbang sebelum mengunduh AlphaDent yang besarnya sekitar 1 GB. Kalau `epoch`
tidak muncul di sini, hentikan dan laporkan.

In [ ]:
import shutil
import torch

staged_last = STAGING / "last.pt"
ckpt = torch.load(staged_last, map_location="cpu", weights_only=False)
args = ckpt["train_args"]

detected = RUN_NAME or args.get("name") or "train"
RUN_DIR = RUNS / detected
weights_dir = RUN_DIR / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)

last_pt = weights_dir / "last.pt"
if not last_pt.exists():
    shutil.move(str(staged_last), last_pt)

print("folder run    :", RUN_DIR)
print("epoch terakhir:", ckpt["epoch"])
print("total epochs  :", args["epochs"])
print("sisa epoch    :", max(args["epochs"] - ckpt["epoch"], 0))
print("data          :", args["data"])
print("imgsz         :", args["imgsz"], "| batch:", args["batch"])
print("project       :", args["project"])
print("versi dibuat  :", ckpt.get("version"))

## Sel 5, pasang Ultralytics sesuai checkpoint

Ini bagian yang sebelumnya gagal. Versi dipin ke `8.3.*`, padahal checkpoint
dibuat dengan `8.4.x`, sehingga argumen seperti `dis`, `dgrad`, `quantize` tidak
dikenal dan Ultralytics melaporkannya sebagai `FileNotFoundError` yang
menyesatkan.

Sekarang versinya dibaca dari checkpoint lalu dipasang persis. Kalau versi itu
tidak tersedia untuk Python 3.13, dipakai versi terbaru dan Sel 6 yang
membersihkan selisihnya.

In [ ]:
import re
import subprocess
import sys

ver = str(ckpt.get("version") or "")
target = ver if re.fullmatch(r"\d+\.\d+\.\d+", ver) else None
print("checkpoint dibuat dengan:", ver or "(tidak tercatat)")


def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1200:])
        print(r.stderr[-1200:])
    return r.returncode == 0


ok = False
if target:
    print("memasang versi yang sama:", target)
    ok = pip_install(f"ultralytics=={target}")

if not ok:
    print("versi tepat tidak tersedia, memasang yang terbaru")
    ok = pip_install("-U", "ultralytics")

assert ok, "pemasangan ultralytics gagal, cek pesan di atas"

import ultralytics

print("terpasang :", ultralytics.__version__)

## Sel 6, selaraskan argumen checkpoint

Jaring pengaman. Kalau versi Ultralytics yang terpasang berbeda dari versi
pembuat checkpoint, argumen yang tidak dikenal dibuang dan tipe boolean
disamakan. Contoh nyata: `multi_scale` dulu float `0.0`, sekarang harus bool.

`last_orig.pt` dibuat sebagai cadangan dan tidak pernah ditimpa.

In [ ]:
import shutil
import torch
from ultralytics.cfg import DEFAULT_CFG_DICT

weights = RUN_DIR / "weights"
current = weights / "last.pt"
backup = weights / "last_orig.pt"

if not backup.exists():
    shutil.copy2(current, backup)
    print("cadangan dibuat :", backup.name)
else:
    shutil.copy2(backup, current)
    print("dipulihkan dari :", backup.name)

ckpt = torch.load(current, map_location="cpu", weights_only=False)
args = ckpt["train_args"]
known = set(DEFAULT_CFG_DICT)

unknown = sorted(k for k in args if k not in known)
for k in unknown:
    args.pop(k)

coerced = []
for k, v in list(args.items()):
    if isinstance(DEFAULT_CFG_DICT.get(k), bool) and not isinstance(v, bool):
        args[k] = bool(v)
        coerced.append(f"{k}: {v!r} -> {args[k]}")

ckpt["train_args"] = args
torch.save(ckpt, current)

print("dibuang  :", unknown or "(tidak ada)")
print("disamakan:", coerced or "(tidak ada)")
print("ditulis  :", current)
print()
print("epoch terakhir:", ckpt["epoch"], "dari", args["epochs"])

## Sel 7, unduh dan ekstrak AlphaDent

1.455 foto gigi DSLR dari HuggingFace, Apache-2.0. Diunduh ulang karena
`/content` hilang setiap sesi berakhir.

In [ ]:
import os
import zipfile

from huggingface_hub import hf_hub_download

ALPHA_DIR = "/content/datasets/alphadent"
os.makedirs(ALPHA_DIR, exist_ok=True)

zip_path = hf_hub_download(
    repo_id="ZFTurbo/AlphaDent",
    filename="AlphaDent.zip",
    repo_type="dataset",
    local_dir="/content/ad_zip",
)
print("zip:", zip_path)

with zipfile.ZipFile(zip_path) as z:
    z.extractall(ALPHA_DIR)


def has_struct(root):
    a = os.path.isdir(os.path.join(root, "images", "train")) and os.path.isdir(
        os.path.join(root, "labels", "train")
    )
    b = os.path.isdir(os.path.join(root, "train", "images")) and os.path.isdir(
        os.path.join(root, "train", "labels")
    )
    return bool(a or b)


AD_ROOT = None
for root, dirs, files in os.walk(ALPHA_DIR):
    if has_struct(root):
        AD_ROOT = root
        break
assert AD_ROOT, "struktur AlphaDent tidak ketemu, cek isi zip"
print("AD_ROOT:", AD_ROOT)

## Sel 8, bangun dataset campuran

Logikanya sama dengan `scripts/prepare_v2_mix.py`. Semua kelas dipetakan ke
`tooth`, dan baris bbox AlphaDent diubah menjadi polygon karena Ultralytics
menolak file yang mencampur keduanya.

Tujuan **wajib** `/content/datasets/mix_v2` karena path itu yang tersimpan di
checkpoint.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

INTRAORAL_DIR = "/content/datasets/intraoral"
os.makedirs(INTRAORAL_DIR, exist_ok=True)

with zipfile.ZipFile(ROOT / "intraoral_yolov8.zip") as z:
    for info in z.infolist():
        name = info.filename.replace("\\", "/")
        target = os.path.join(INTRAORAL_DIR, name)
        if name.endswith("/"):
            os.makedirs(target, exist_ok=True)
            continue
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(info) as src, open(target, "wb") as dst:
            dst.write(src.read())
print("intraoral:", os.listdir(os.path.join(INTRAORAL_DIR, "intraoral")))

JOBS = [
    ("io_", Path(INTRAORAL_DIR) / "intraoral"),
    ("ad_", Path(AD_ROOT)),
]
SPLITS = [("train", "train"), ("valid", "valid"), ("test", "valid")]


def pick_dirs(root, split):
    imgs = [p for p in (root / split / "images", root / "images" / split) if p.is_dir()]
    lbls = [p for p in (root / split / "labels", root / "labels" / split) if p.is_dir()]
    return (imgs[0] if imgs else None), (lbls[0] if lbls else None)


def normalize_row(line):
    parts = line.split()
    if len(parts) < 2:
        return None
    coords = parts[1:]
    if len(coords) == 4:
        cx, cy, bw, bh = (float(v) for v in coords)
        x1, y1, x2, y2 = cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2
        corners = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
        return "0 " + " ".join(f"{v:.6f}" for pt in corners for v in pt)
    if len(coords) >= 6 and len(coords) % 2 == 0:
        return "0 " + " ".join(coords)
    return None


counts = {"train": 0, "valid": 0}
dropped = 0
for prefix, root in JOBS:
    for split_src, split_dst in SPLITS:
        imgs, lbls = pick_dirs(root, split_src)
        if imgs is None or lbls is None:
            continue
        (MIX_DIR / split_dst / "images").mkdir(parents=True, exist_ok=True)
        (MIX_DIR / split_dst / "labels").mkdir(parents=True, exist_ok=True)
        for img in sorted(imgs.iterdir()):
            if img.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            lbl = lbls / (img.stem + ".txt")
            if not lbl.exists():
                continue
            rows = []
            for line in lbl.read_text().splitlines():
                t = line.strip()
                if not t:
                    continue
                row = normalize_row(t)
                if row is None:
                    dropped += 1
                    continue
                rows.append(row)
            if not rows:
                continue
            shutil.copy(img, MIX_DIR / split_dst / "images" / f"{prefix}{img.name}")
            (MIX_DIR / split_dst / "labels" / f"{prefix}{img.stem}.txt").write_text(
                "\n".join(rows) + "\n"
            )
            counts[split_dst] += 1

(MIX_DIR / "data.yaml").write_text(
    "train: train/images\nval: valid/images\nnc: 1\nnames: ['tooth']\n"
)
print("merged:", counts, "->", MIX_DIR)
if dropped:
    print("baris label dibuang:", dropped)
assert (MIX_DIR / "data.yaml").exists()

## Sel 9, lanjutkan latihan

Tanpa argumen tambahan. `resume=True` membaca ulang semua pengaturan dari
checkpoint, termasuk folder output, jadi latihan lanjut di folder yang sama dan
tidak membuat `train-2` baru.

Latihan bisa berhenti sendiri sebelum epoch terakhir karena `patience=20`. Itu
normal, `best.pt` tetap tersimpan dari epoch terbaik.

In [ ]:
import os

os.environ["WANDB_MODE"] = "disabled"
os.environ["COMET_MODE"] = "disabled"

from ultralytics import YOLO

data_yaml = Path(ckpt["train_args"]["data"])
assert data_yaml.exists(), f"dataset tidak ada: {data_yaml}, ulangi Sel 8"
assert (RUN_DIR / "weights" / "last.pt").exists(), "last.pt hilang, ulangi Sel 4"

model = YOLO(str(RUN_DIR / "weights" / "last.pt"))
model.train(resume=True)

## Sel 10, ekspor hasil

In [ ]:
import shutil
from pathlib import Path

out = Path("/kaggle/working/export")
out.mkdir(parents=True, exist_ok=True)

for name in ("best.pt", "last.pt"):
    src = RUN_DIR / "weights" / name
    if src.exists():
        shutil.copy(src, out / name)
        print(name, "->", round(src.stat().st_size / 1e6, 1), "MB")

csv = RUN_DIR / "results.csv"
if csv.exists():
    shutil.copy(csv, out / "results.csv")

print()
print("folder:", out)
print("Unduh best.pt, lalu taruh di backend/models/seg/best.pt")

## Setelah selesai

Unduh `best.pt` dari panel **Output** di kanan, lalu taruh di
`backend/models/seg/best.pt`.

## Kalau sesi mati lagi

Ulangi Sel 0 sampai Sel 4. Ambil `last.pt` terbaru dari Output notebook ini
kalau Drive belum terbarui, unggah ulang ke Drive lalu pakai ID yang baru.
Sel 7 dan Sel 8 perlu diulang karena `/content` hilang.

## Kalau masih menolak

| Gejala | Sebab | Tindakan |
|---|---|---|
| `TypeError: 'multi_scale=...' invalid type` | Sel 6 belum jalan | jalankan Sel 6 |
| `SyntaxError: 'dis' is not a valid YOLO argument` | versi Ultralytics salah | ulangi Sel 5 dan Sel 6 |
| `FileNotFoundError` data.yaml | Sel 8 belum selesai | ulangi Sel 8, pastikan ada `merged:` |
| `cuda: False` | Accelerator belum GPU | ubah di Settings, restart sesi |